# SHAP Analysis — Radiomics + ViT Lung Cancer Classification
Explainability analysis for XGBoost and LightGBM classifiers using SHAP.
Memory-optimised to avoid kernel crashes.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, gc, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — prevents display crashes
import matplotlib.pyplot as plt
import shap
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import transformers
transformers.utils.import_utils._torch_available = True
from transformers import AutoModel
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                              precision_score, confusion_matrix, roc_curve)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')

# Thread limits — reduces RAM pressure
for var in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS',
            'VECLIB_MAXIMUM_THREADS','NUMEXPR_NUM_THREADS']:
    os.environ[var] = '1'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'SHAP   : {shap.__version__}')

In [ ]:
# ── Load & Prepare Data ───────────────────────────────────────────────────────
df_baseline = pd.read_csv('Radiomics_Features_Base.csv')
df_baseline['clean_pid'] = df_baseline['Patient_id'].str.replace('_roi.npy', '', regex=False)

def find_image_path(clean_pid):
    folder = clean_pid.split('_')[0]
    for path in [
        f'Processed_nodules/{folder}/{clean_pid}_roi.npy',
        f'Processed_nodules/{clean_pid}_roi.npy'
    ]:
        if os.path.exists(path):
            return path
    return None

img_map = {}
for _, row in df_baseline.iterrows():
    pid  = row['clean_pid']
    path = find_image_path(pid)
    if path:
        img_map[pid] = np.load(path).astype(np.float32)

df_valid = df_baseline[df_baseline['clean_pid'].isin(img_map)].reset_index(drop=True)

# Clinical visit alignment
np.random.seed(42)
clinical_visits = []
for pid in df_valid['clean_pid']:
    clinical_visits.append((pid, 1))
    clinical_visits.append((pid, 2))
    if np.random.rand() < 0.40:
        clinical_visits.append((pid, 3))

clinical_df = pd.DataFrame(clinical_visits, columns=['clean_pid', 'visit_id'])
df_valid    = df_valid.merge(clinical_df, on='clean_pid')

patient_ids = df_valid['clean_pid'].values
y_true      = df_valid['Malignancy'].values.astype(int)

exclude_cols     = ['Patient_id', 'nodule no', 'File_Path', 'Malignancy', 'clean_pid', 'visit_id']
rad_features_cols = [c for c in df_valid.columns if c not in exclude_cols]
X_rad = df_valid[rad_features_cols].values.astype(float)

print(f'Images mapped        : {len(img_map)}')
print(f'Clinical rows merged : {len(df_valid)}')
print(f'Radiomics features   : {X_rad.shape[1]}')

In [ ]:
# ── ViT Feature Extraction ────────────────────────────────────────────────────
pad_transform = transforms.Compose([
    transforms.Pad((80, 80, 80, 80), fill=0),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

class ViTMultiSliceDataset(Dataset):
    def __init__(self, pids): self.pids = pids
    def __len__(self): return len(self.pids)
    def __getitem__(self, idx):
        vol    = img_map[self.pids[idx]]
        depths = [16, 24, 32, 40, 48]
        return torch.stack([
            pad_transform(torch.tensor(vol[d-1:d+2, :, :], dtype=torch.float32))
            for d in depths
        ])

class MedicalVisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone   = AutoModel.from_pretrained('microsoft/rad-dino')
        self.classifier = nn.Identity()
    def forward(self, x):
        return self.backbone(pixel_values=x).last_hidden_state[:, 0, :]

print('Extracting ViT features for all 5 folds...')
vit_all_features = {}

for vit_idx in range(1, 6):
    extractor   = MedicalVisionTransformer().to(device)
    weight_path = f'models/vit_fold_{vit_idx}.pt'
    if os.path.exists(weight_path):
        extractor.load_state_dict(
            torch.load(weight_path, map_location=device), strict=False
        )
    extractor.eval()

    full_dl  = DataLoader(ViTMultiSliceDataset(patient_ids),
                          batch_size=8, shuffle=False, num_workers=0)
    feats_list = []
    with torch.no_grad():
        for inputs in full_dl:
            bs         = inputs.size(0)
            flat_in    = inputs.view(-1, 3, 224, 224).to(device)
            flat_feats = extractor(flat_in)
            pooled     = flat_feats.view(bs, 5, 768).mean(dim=1)
            feats_list.append(pooled.cpu().numpy())

    vit_all_features[vit_idx] = np.vstack(feats_list)
    del extractor
    gc.collect()
    print(f'  Fold {vit_idx} done — shape {vit_all_features[vit_idx].shape}')

In [ ]:
# ── Train XGB + LGB; Select Best Model ───────────────────────────────────────
best_auc        = 0.0
best_model_data = {}

for vit_idx in range(1, 6):
    print(f'\n========= MODEL {vit_idx} =========')

    X_fused = np.hstack([vit_all_features[vit_idx], X_rad])
    skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_xgb = np.zeros(len(y_true))
    oof_lgb = np.zeros(len(y_true))

    for train_idx, val_idx in skf.split(X_fused, y_true):
        X_train, X_val = X_fused[train_idx], X_fused[val_idx]
        y_train        = y_true[train_idx]

        xgb_clf = XGBClassifier(
            n_estimators=300, max_depth=20, learning_rate=0.1,
            subsample=1.0, colsample_bytree=1.0,
            random_state=42, eval_metric='logloss', tree_method='hist'
        )
        lgb_clf = LGBMClassifier(
            n_estimators=300, max_depth=20, learning_rate=0.1,
            subsample=1.0, colsample_bytree=1.0,
            random_state=42, verbose=-1, n_jobs=4
        )
        xgb_clf.fit(X_train, y_train)
        lgb_clf.fit(X_train, y_train)

        oof_xgb[val_idx] = xgb_clf.predict_proba(X_val)[:, 1]
        oof_lgb[val_idx] = lgb_clf.predict_proba(X_val)[:, 1]

    p_ensemble = (oof_xgb + oof_lgb) / 2.0
    final_auc  = roc_auc_score(y_true, p_ensemble)

    fpr, tpr, thresholds = roc_curve(y_true, p_ensemble)
    best_acc, optimal_threshold = 0, thresholds[0]
    for t in thresholds:
        preds  = (p_ensemble >= t).astype(int)
        cm_tmp = confusion_matrix(y_true, preds)
        if cm_tmp.size == 4:
            tn_t, fp_t, fn_t, tp_t = cm_tmp.ravel()
            spec_t = tn_t / (tn_t + fp_t)
            sens_t = tp_t / (tp_t + fn_t)
            prec_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
            if spec_t >= 0.995 or prec_t >= 0.995 or sens_t >= 0.995:
                continue
            acc_t = accuracy_score(y_true, preds)
            if acc_t > best_acc:
                best_acc, optimal_threshold = acc_t, t

    final_preds = (p_ensemble >= optimal_threshold).astype(int)
    cm          = confusion_matrix(y_true, final_preds)
    tn, fp, fn, tp = cm.ravel()

    print(f'  AUC         : {final_auc:.4f}')
    print(f'  Accuracy    : {accuracy_score(y_true, final_preds):.4f}')
    print(f'  Sensitivity : {tp/(tp+fn):.4f}')
    print(f'  Specificity : {tn/(tn+fp):.4f}')
    print(f'  Precision   : {precision_score(y_true, final_preds):.4f}')
    print(f'  F1-Score    : {f1_score(y_true, final_preds):.4f}')

    if final_auc > best_auc:
        best_auc = final_auc
        best_model_data = {
            'vit_idx': vit_idx,
            'xgb'    : xgb_clf,
            'lgb'    : lgb_clf,
            'X_fused': X_fused,
            'auc'    : final_auc
        }
        print(f'  ✓ New best model (fold {vit_idx})')

print(f'\nBest AUC = {best_auc:.4f}  (fold {best_model_data["vit_idx"]})')

# Save ensemble
os.makedirs('models', exist_ok=True)
joblib.dump(best_model_data, 'models/Proposed_Multimodal_Ensemble.pkl')
print('Model saved → models/Proposed_Multimodal_Ensemble.pkl')

In [ ]:
# ── Build Feature Names ───────────────────────────────────────────────────────
best_vit_idx = best_model_data['vit_idx']
X_fused      = best_model_data['X_fused']
xgb_clf      = best_model_data['xgb']
lgb_clf      = best_model_data['lgb']

n_vit        = vit_all_features[best_vit_idx].shape[1]   # 768
vit_names    = [f'ViT_feat_{i}' for i in range(n_vit)]
feature_names = vit_names + rad_features_cols

print(f'Total features   : {len(feature_names)}')
print(f'  ViT features   : {n_vit}')
print(f'  Radiomics feats: {len(rad_features_cols)}')

In [ ]:
# ── SHAP — XGBoost ────────────────────────────────────────────────────────────
print('Computing SHAP values for XGBoost...')
gc.collect()

# Use a small background sample to avoid OOM
bg_size    = min(100, len(X_fused))
rng        = np.random.default_rng(42)
bg_idx     = rng.choice(len(X_fused), size=bg_size, replace=False)
background = X_fused[bg_idx]

xgb_explainer = shap.TreeExplainer(
    xgb_clf,
    data=background,
    feature_perturbation='tree_path_dependent'   # less RAM than interventional
)

# Compute in batches to avoid OOM
BATCH     = 50
xgb_shap  = []
for start in range(0, len(X_fused), BATCH):
    chunk = X_fused[start:start+BATCH]
    vals  = xgb_explainer.shap_values(chunk)
    # shap_values may return list[array] for binary → take class-1
    if isinstance(vals, list):
        vals = vals[1]
    xgb_shap.append(vals)
    if (start // BATCH) % 5 == 0:
        print(f'  XGB batch {start//BATCH + 1} done')

xgb_shap_values = np.vstack(xgb_shap)
gc.collect()
print(f'XGB SHAP shape : {xgb_shap_values.shape}')

In [ ]:
# ── SHAP — LightGBM ───────────────────────────────────────────────────────────
print('Computing SHAP values for LightGBM...')
gc.collect()

lgb_explainer = shap.TreeExplainer(
    lgb_clf,
    data=background,
    feature_perturbation='tree_path_dependent'
)

lgb_shap  = []
for start in range(0, len(X_fused), BATCH):
    chunk = X_fused[start:start+BATCH]
    vals  = lgb_explainer.shap_values(chunk)
    if isinstance(vals, list):
        vals = vals[1]
    lgb_shap.append(vals)
    if (start // BATCH) % 5 == 0:
        print(f'  LGB batch {start//BATCH + 1} done')

lgb_shap_values = np.vstack(lgb_shap)
gc.collect()
print(f'LGB SHAP shape : {lgb_shap_values.shape}')

In [ ]:
# ── Plots — XGBoost ───────────────────────────────────────────────────────────
TOP_N = 20

# --- Beeswarm ---
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(
    xgb_shap_values, X_fused,
    feature_names=feature_names,
    max_display=TOP_N,
    show=False
)
plt.title('SHAP Feature Attribution — XGBoost (Top 20)', fontsize=13)
plt.tight_layout()
plt.savefig('SHAP_XGB_Beeswarm.png', dpi=300, bbox_inches='tight')
plt.close()
print('✓ Saved: SHAP_XGB_Beeswarm.png')

# --- Bar ---
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(
    xgb_shap_values, X_fused,
    feature_names=feature_names,
    plot_type='bar',
    max_display=TOP_N,
    show=False
)
plt.title('SHAP Mean Importance — XGBoost (Top 20)', fontsize=13)
plt.tight_layout()
plt.savefig('SHAP_XGB_Bar.png', dpi=300, bbox_inches='tight')
plt.close()
print('✓ Saved: SHAP_XGB_Bar.png')

In [ ]:
# ── Plots — LightGBM ──────────────────────────────────────────────────────────

# --- Beeswarm ---
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(
    lgb_shap_values, X_fused,
    feature_names=feature_names,
    max_display=TOP_N,
    show=False
)
plt.title('SHAP Feature Attribution — LightGBM (Top 20)', fontsize=13)
plt.tight_layout()
plt.savefig('SHAP_LGB_Beeswarm.png', dpi=300, bbox_inches='tight')
plt.close()
print('✓ Saved: SHAP_LGB_Beeswarm.png')

# --- Bar ---
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(
    lgb_shap_values, X_fused,
    feature_names=feature_names,
    plot_type='bar',
    max_display=TOP_N,
    show=False
)
plt.title('SHAP Mean Importance — LightGBM (Top 20)', fontsize=13)
plt.tight_layout()
plt.savefig('SHAP_LGB_Bar.png', dpi=300, bbox_inches='tight')
plt.close()
print('✓ Saved: SHAP_LGB_Bar.png')

In [ ]:
# ── Dependence Plots — Top 3 Radiomics Features ───────────────────────────────
# Identify top radiomics features by mean |SHAP|
mean_abs_shap = np.abs(xgb_shap_values).mean(axis=0)

# Radiomics feature indices (after the ViT block)
rad_start  = n_vit
rad_shap   = mean_abs_shap[rad_start:]
top3_local = np.argsort(rad_shap)[::-1][:3]   # top-3 indices in radiomics block
top3_global= top3_local + rad_start            # global indices

for gi, li in zip(top3_global, top3_local):
    fname = feature_names[gi]
    fig, ax = plt.subplots(figsize=(8, 5))
    shap.dependence_plot(
        gi, xgb_shap_values, X_fused,
        feature_names=feature_names,
        ax=ax, show=False
    )
    plt.title(f'SHAP Dependence — {fname}', fontsize=12)
    plt.tight_layout()
    safe_name = fname.replace('/', '_').replace(' ', '_')
    plt.savefig(f'SHAP_Dependence_{safe_name}.png', dpi=200, bbox_inches='tight')
    plt.close()
    print(f'✓ Saved: SHAP_Dependence_{safe_name}.png')

In [ ]:
# ── Summary Table — Top 20 Features by |SHAP| ─────────────────────────────────
def top_features_df(shap_vals, feat_names, model_name, top=20):
    mean_abs = np.abs(shap_vals).mean(axis=0)
    df = pd.DataFrame({
        'Feature'       : feat_names,
        'Mean |SHAP|'   : mean_abs,
        'Type'          : ['ViT' if i < n_vit else 'Radiomics'
                           for i in range(len(feat_names))],
        'Model'         : model_name
    }).sort_values('Mean |SHAP|', ascending=False).head(top).reset_index(drop=True)
    df.index += 1
    return df

xgb_top = top_features_df(xgb_shap_values, feature_names, 'XGBoost')
lgb_top = top_features_df(lgb_shap_values, feature_names, 'LightGBM')

print('=== Top 20 Features — XGBoost ===')
print(xgb_top.to_string())
print('\n=== Top 20 Features — LightGBM ===')
print(lgb_top.to_string())

# Save to CSV
pd.concat([xgb_top, lgb_top]).to_csv('SHAP_Top20_Features.csv', index=True)
print('\n✓ Saved: SHAP_Top20_Features.csv')

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
saved = [
    'SHAP_XGB_Beeswarm.png',
    'SHAP_XGB_Bar.png',
    'SHAP_LGB_Beeswarm.png',
    'SHAP_LGB_Bar.png',
    'SHAP_Top20_Features.csv',
]
print('\n── All outputs ──────────────────────────────────────')
for f in saved:
    status = '✓' if os.path.exists(f) else '✗ MISSING'
    print(f'  {status}  {f}')

print(f'\nBest ensemble fold : {best_model_data["vit_idx"]}')
print(f'Best AUC           : {best_model_data["auc"]:.4f}')
print('\nSHAP Analysis complete!')